The code below is to get the data and turn it into csv files

In [10]:
import pandas as pd
import numpy as np
import yfinance as yf

pd.set_option('display.max_columns', None)

start = "2018-01-01"
end = "2024-01-01"
tickers = ["SPY","QQQ","TLT"]

for ticker in tickers:
    df = yf.download(ticker,start,end)

    #isNull() makes the df into 1 and 0 where 1 is null. The first sum gets the number of 1 in each column.The second sum sums the number in each column
    num_missing = df.isnull().sum().sum()
    print(f"Total Missing: {num_missing}")

    if num_missing>0:
        #This fills the missing values. ffill() fills it with prev value. bfill() fills with value ahead if there is nothing prev.
        df = df.ffill().bfill()
    
    #Gets rid of the multiIndex which can be annoying. We get rid 2nd layer of the columns
    if isinstance(df.columns,pd.MultiIndex):
        df.columns = df.columns.droplevel(1)

    
    filename = f"{ticker}_raw.csv"
    df.to_csv(filename)






[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Total Missing: 0
Total Missing: 0
Total Missing: 0


In [11]:
def get_indicators(df):
    """
    Takes a raw OHLCV dataframe, shifts it to avoid lookahead bias, 
    and adds RSI, MACD, and Bollinger Bands.
    """

    #shifts the day one down. So for example tuesday only has the clost information available on monday. No look ahead bias
    df['Past_close'] = df['Close'].shift(1)

    #Relative Strength Index(14 day rolling window)
    #Delta is a df that is the current day subtracted from previous day
    delta = df['Past_close'].diff()
    gain = (delta.where(delta>0,0)).rolling(window=14).mean()
    loss = (-delta.where(delta<0,0)).rolling(window=14).mean()
    rs = gain/loss
    df['RSI'] = 100 - (100 / (1 + rs))


    #Bollinger Bands(Usually a 20-day rolling mean, plus/minus 2 standard deviations)
    df['BB_Mid'] = df['Past_close'].rolling(window=20).mean()
    bb_std = df['Past_close'].rolling(window=20).std()
    df['BB_Upper'] = df['BB_Mid'] + (2 * bb_std)
    df['BB_Lower'] = df['BB_Mid'] - (2 * bb_std)


    #Moving Average Convergence Divergence(Difference between 12-day EMA and 26-day EMA)
    ema_12 = df['Past_close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['Past_close'].ewm(span=26,adjust=False).mean()
    df['MACD'] = ema_12 - ema_26
    #When MACD is above the signal line, it means the trend is going upward. When MACD is below the signal line it means trend is going down
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    #Gets rid of NaN
    df.dropna(inplace=True) 

    return df




The code below is to get the data seperated for training

In [17]:
import pickle

#To hold all of our dataframes
all_dfs = []

tickers = ["SPY","QQQ","TLT"]

for tick in tickers:
    df_raw = pd.read_csv(f"{ticker}_raw.csv", index_col=0, parse_dates=True)
    df_engineered = get_indicators(df_raw)

    df_engineered['Ticker'] = tick

    #These are for when training the model
    df_engineered['weight'] = 0.0
    df_engineered['regime_prob'] = 0.0

    all_dfs.append(df_engineered)

all_dfs = pd.concat(all_dfs)

#This below will be the training, validating, and testing sets below
#Train is 2018-2021, Validate will be 2022-2023, test will be 2023-2024
train_data = all_dfs.loc[(all_dfs.index<'2022-01-01')]
val_data = all_dfs.loc[(all_dfs.index<'2023-01-01') & (all_dfs.index>='2022-01-01')]
test_data =  all_dfs.loc[(all_dfs.index>='2023-01-01')]

display(val_data)
print(f"Train Shape: {train_data.shape}")
#There are 753 rows because three tickers 251*3 = 753
print(f"Val Shape: {val_data.shape}")
print(f"Test Shape: {test_data.shape}")

#Turn to dict so it is easier for RL gym env
output = {
    'train': train_data,
    'val': val_data,
    'test':test_data
}

#Turn to pickle because it is dict
with open('engineered.pkl', 'wb') as f:
    pickle.dump(output, f)







    





,Close,High,Low,Open,Volume,Past_close,RSI,BB_Mid,BB_Upper,BB_Lower,MACD,MACD_signal,Ticker,weight,regime_prob
Date,,,,,,,,,,,,,,,
2022-01-03,123.909172,126.176117,123.891995,125.721011,33860400,127.249466,48.143584,128.400329,131.271046,125.529612,-0.117075,0.194490,SPY,0.0,0.0
2022-01-04,123.393951,123.763196,122.569615,123.316672,21996400,123.909172,28.359514,127.976297,130.871056,125.081538,-0.406134,0.074365,SPY,0.0,0.0
2022-01-05,122.724144,123.788926,122.543826,123.771748,20911700,123.393951,27.918713,127.617422,130.909852,124.324992,-0.669077,-0.074323,SPY,0.0,0.0
2022-01-06,123.041893,123.170705,122.183202,122.440812,18996400,122.724144,29.179548,127.277806,131.103524,123.452089,-0.920893,-0.243637,SPY,0.0,0.0
2022-01-07,122.157448,122.998975,121.556366,122.904518,18756800,123.041893,30.642298,127.066013,131.335054,122.796972,-1.082343,-0.411378,SPY,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-23,89.788841,90.360124,89.665789,90.157979,15408900,91.124779,40.093147,92.754985,96.881562,88.628408,0.994747,1.576328,TLT,0.0,0.0
2022-12-27,88.013474,88.830855,87.899218,88.321089,26475700,89.788841,40.033699,92.744091,96.902245,88.585938,0.705509,1.402165,TLT,0.0,0.0
2022-12-28,87.494926,88.575975,87.319142,88.461720,17302900,88.013474,30.256919,92.631746,97.175942,88.087550,0.329233,1.187578,TLT,0.0,0.0


Train Shape: (2964, 15)
Val Shape: (753, 15)
Test Shape: (750, 15)
